# Assignment 5 — Many-Objective Optimisation with JUSTICE (Local Run)

**Course:** EPA141A Model-Based Decision Making — Delft University of Technology  
**Model:** JUSTICE  
**Actor 15 — Japan & South Korea (Bloc 4, High Ambition Coalition)**

---

## Learning Outcomes

After completing this assignment you will be able to:
1. Load and inspect the optimisation configuration produced in Assignment 4 (`config/config_student.json`).
2. Run a many-objective evolutionary algorithm (**GenerationalBorg**) on JUSTICE via `run_optimization_local.py`.
3. Load, sanity-check, and pool the Pareto-front CSVs into the **reference set** that Assignments 6, 7 and 8 consume.

---

## Background

Assignment 4 defined the optimisation problem: 244 RBF decision variables, four objectives, and the epsilon values that control archive resolution. It saved those choices to `config/config_student.json`. This notebook runs the actual Pareto search against that configuration.

The optimiser is **GenerationalBorg**, a self-adaptive MOEA that applies six variation operators (SBX, PCX, DE, UNDX, SPX, UM) at once and continuously re-weights them toward whichever is currently improving the archive. That self-tuning is what makes it suitable for a high-dimensional, non-convex search like our 244-parameter RBF space.

Each function evaluation runs JUSTICE once with one candidate policy (one set of 244 RBF parameters) and returns the four objective values. Non-dominated candidates accumulate in an **epsilon-archive**: a solution is only kept if it improves on the archive by at least the relevant ε in at least one objective. Because the search is stochastic, a single run can miss parts of the front, so the full protocol runs five independent seeds and pools their archives by epsilon-dominance into a **combined reference set**. That reference set is the object Assignments 6 (convergence), 7 (pathways) and 8 (robustness) all read.

### What this means for Japan & South Korea

We optimise under our **primary (Utilitarian) lens** chosen in Assignment 4: it sums per-capita welfare across all 57 regions and lets a technology-led, outcome-conditioned policy score well as long as the long-run target is met. The RBF policy itself is the modelling form of our **technology-neutrality** demand (fix the temperature outcome, let the route adapt). The four objectives map straight onto the mandate: `welfare` (efficiency), `fraction_above_threshold` (our 2°C climate test), `welfare_loss_damage` (the channel that hurts exposed regions such as `rcam`, `rjan57` during any overshoot), and `welfare_loss_abatement` (the mitigation price, which as technology exporters we want kept low globally). We do **not** pick a single recommended policy here — that decision is deferred to the robustness analysis in Assignment 8, where our overshoot-tolerance and worst-off concerns can be tested across scenarios.

---

## Starting point — your Assignment 4 configuration

| Mode | Command (run from a terminal) |
|---|---|
| Smoke test (< 5 min, validates the pipeline) | `python run_optimization_local.py --nfe 500 --seeds 9845531 --config config/config_student.json` |
| Single full seed (~1–3 h) | `python run_optimization_local.py --nfe 50000 --seeds 9845531 --config config/config_student.json` |
| Full 5-seed run (overnight, the submitted reference set) | `python run_optimization_local.py --nfe 50000 --config config/config_student.json` |

> The smoke test exists only to confirm the config and paths are correct. The reference set we actually analyse downstream should come from the full 5-seed, 50 000-NFE run.


---

## Step 1 — Inspect the configuration

The optimisation is governed entirely by `config/config_student.json`, written by Assignment 4. We load it and print each key with a short explanation, so the run is reproducible from a single file.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, sys, json
import numpy as np
import pandas as pd

# ── Paths (siblings under the parent of this notebook dir) ────────────────────
#   <parent>/<this notebook dir>
#   <parent>/JUSTICE-main      <- model source
#   <parent>/config            <- config_student.json (written by Assignment 4)
#   <parent>/results           <- MOEA output (written by this notebook)
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath(".")

_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))
CONFIG_PATH   = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../config/config_student.json"))
RESULTS_ROOT  = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../results"))
os.makedirs(RESULTS_ROOT, exist_ok=True)

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)
os.chdir(_JUSTICE_ROOT)

# ── Load the Assignment 4 configuration ──────────────────────────────────────
with open(CONFIG_PATH) as fh:
    cfg = json.load(fh)

explanations = {
    "start_year":                       "Simulation start year",
    "end_year":                         "Simulation end year",
    "data_timestep":                    "Years between raw input data points",
    "timestep":                         "Model integration timestep (years)",
    "emission_control_start_year":      "First year ECR can exceed zero",
    "n_rbfs":                           "Number of RBFs (fixed architecture: 4)",
    "n_inputs":                         "RBF input signals (scaled temperature + delta-temperature)",
    "epsilons":                         "Archive granularity [welfare, frac_above, wl_damage, wl_abatement]",
    "temperature_year_of_interest":     "Year for the 2C threshold-fraction evaluation",
    "reference_ssp_rcp_scenario_index": "Reference scenario index (2 = SSP2-RCP4.5, our central overshoot reference)",
}

print(f"Configuration: {CONFIG_PATH}\n")
for k, v in cfg.items():
    print(f"  {k:<40s}  {str(v):<18}  # {explanations.get(k, '')}")

# ── Guard: this notebook assumes the Assignment 4 choices ────────────────────
assert cfg["n_rbfs"] == 4, "Expected n_rbfs == 4 (fixed RBF architecture from Assignment 4)"
assert cfg["reference_ssp_rcp_scenario_index"] == 2, (
    "Expected reference scenario index 2 (SSP2-4.5). "
    "Re-run Assignment 4 to regenerate the config if this fails."
)
print("\nConfig matches the Assignment 4 problem formulation.")


Configuration: /Users/stijnkeukens/epa141a-main/config/config_student.json

  start_year                                2015                # Simulation start year
  end_year                                  2300                # Simulation end year
  data_timestep                             5                   # Years between raw input data points
  timestep                                  1                   # Model integration timestep (years)
  emission_control_start_year               2025                # First year ECR can exceed zero
  n_rbfs                                    4                   # Number of RBFs (fixed architecture: 4)
  n_inputs                                  2                   # RBF input signals (scaled temperature + delta-temperature)
  epsilons                                  [0.1, 0.25, 10.0, 10.0]  # Archive granularity [welfare, frac_above, wl_damage, wl_abatement]
  temperature_year_of_interest              2100                # Year for the 2C 

In [2]:
from justice.util.data_loader import DataLoader

# Read the RBF architecture straight from the config (no n_inputs+2 shortcut —
# n_rbfs is a fixed design choice of 4, exactly as set in Assignment 4).
n_inputs  = cfg["n_inputs"]
n_rbfs    = cfg["n_rbfs"]                 # = 4
n_regions = len(DataLoader().REGION_LIST) # = 57

n_centers = n_rbfs * n_inputs
n_radii   = n_rbfs * n_inputs
n_weights = n_rbfs * n_regions
n_total   = n_centers + n_radii + n_weights

print("RBF decision-variable summary:")
print(f"  n_rbfs    = {n_rbfs}")
print(f"  n_inputs  = {n_inputs}")
print(f"  n_regions = {n_regions}")
print("  " + "-" * 33)
print(f"  Centers   = {n_rbfs} x {n_inputs} = {n_centers:4d}   range [-1, 1]")
print(f"  Radii     = {n_rbfs} x {n_inputs} = {n_radii:4d}   range [ 0, 1]")
print(f"  Weights   = {n_rbfs} x {n_regions} = {n_weights:4d}   range [ 0, 1]")
print("  " + "-" * 33)
print(f"  TOTAL                = {n_total}")
assert n_total == 244, f"Expected 244 decision variables, got {n_total}"

print("\nObjectives and epsilon (archive) resolution:")
OBJ_NAMES = ["welfare", "fraction_above_threshold", "welfare_loss_damage", "welfare_loss_abatement"]
OBJ_GLOSS = {
    "welfare":                  "Utilitarian welfare-loss magnitude (lower = better)",
    "fraction_above_threshold": "fraction of FaIR ensemble above 2C in 2100 (our climate test)",
    "welfare_loss_damage":      "loss from climate damage (hits exposed regions rcam, rjan57)",
    "welfare_loss_abatement":   "loss from abatement cost (the mitigation price)",
}
for name, eps in zip(OBJ_NAMES, cfg["epsilons"]):
    print(f"  {name:<26s}  eps = {eps:<6}  # {OBJ_GLOSS[name]}")


RBF decision-variable summary:
  n_rbfs    = 4
  n_inputs  = 2
  n_regions = 57
  ---------------------------------
  Centers   = 4 x 2 =    8   range [-1, 1]
  Radii     = 4 x 2 =    8   range [ 0, 1]
  Weights   = 4 x 57 =  228   range [ 0, 1]
  ---------------------------------
  TOTAL                = 244

Objectives and epsilon (archive) resolution:
  welfare                     eps = 0.1     # Utilitarian welfare-loss magnitude (lower = better)
  fraction_above_threshold    eps = 0.25    # fraction of FaIR ensemble above 2C in 2100 (our climate test)
  welfare_loss_damage         eps = 10.0    # loss from climate damage (hits exposed regions rcam, rjan57)
  welfare_loss_abatement      eps = 10.0    # loss from abatement cost (the mitigation price)


---

## Step 2 — Run the optimisation

The MOEA is implemented in `run_optimization_local.py`. Every function evaluation builds one RBF policy from 244 parameters, runs JUSTICE forward to 2300 under the reference scenario (SSP2-4.5) across the local FaIR subset, and returns our four objectives; GenerationalBorg keeps the epsilon-non-dominated ones.

The cell below launches the run. Keep `NFE = 500` / one seed for a quick pipeline check; switch to `NFE = 50000` with all five seeds for the reference set we actually submit. Long runs are better started from a terminal so they survive a closed notebook.


In [7]:
# ── Configure here ───────────────────────────────────────────────────────────
RUN_NOW     = True        # False -> only print the command, run it yourself in a terminal
NFE         = 50000         # smoke test; use 50_000 for the real run
SEEDS       = [9845531]   # full protocol: [9845531, 1644652, 3569126, 6075612, 521475]
N_ENSEMBLES = 10          # 10 well-distributed FaIR members (matches Assignment 4)
N_PROCESSES = None        # None -> auto-detect (cpu_count - 1)
# ─────────────────────────────────────────────────────────────────────────────

# Locate run_optimization_local.py without assuming a fixed folder name
_candidates = [
    _NOTEBOOK_DIR,
    os.path.join(_NOTEBOOK_DIR, "..", "model_answers_ema"),
    os.path.join(_NOTEBOOK_DIR, "..", "assignments_ema"),
    os.path.join(_JUSTICE_ROOT, ".."),
    _JUSTICE_ROOT,
]
script_path = next(
    (os.path.normpath(os.path.join(d, "run_optimization_local.py"))
     for d in _candidates
     if os.path.exists(os.path.join(d, "run_optimization_local.py"))),
    None,
)

seeds_str  = " ".join(str(s) for s in SEEDS)
n_proc_arg = f"--n_processes {N_PROCESSES}" if N_PROCESSES else ""

if script_path is None:
    print("run_optimization_local.py not found. Searched:")
    for d in _candidates:
        print("  ", os.path.normpath(d))
else:
    cmd = (
        f'python "{script_path}" '
        f"--nfe {NFE} "
        f"--seeds {seeds_str} "
        f"--n_ensembles {N_ENSEMBLES} "
        f'--output_dir "{RESULTS_ROOT}" '
        f'--config "{CONFIG_PATH}" '
        + n_proc_arg
    ).strip()

    print("Command:")
    print(" ", cmd)
    if RUN_NOW:
        print("\nRunning ... (blocks the kernel until the run finishes)")
        ret = os.system(cmd)
        print(f"\nExit code: {ret}  ({'OK' if ret == 0 else 'ERROR'})")
    else:
        print("\nRUN_NOW is False — copy the command into a terminal to run it.")


Command:
  python "/Users/stijnkeukens/epa141a-main/model_answers_ema/run_optimization_local.py" --nfe 50000 --seeds 9845531 --n_ensembles 10 --output_dir "/Users/stijnkeukens/epa141a-main/results" --config "/Users/stijnkeukens/epa141a-main/config/config_student.json"

Running ... (blocks the kernel until the run finishes)
JUSTICE — Local MOEA Optimisation  (Assignment 5)
  Welfare function  : UTILITARIAN
  Config            : /Users/stijnkeukens/epa141a-main/config/config_student.json
  NFE per seed      : 50,000
  Seeds (1)        : [9845531]
  FAIR ensembles    : 10  (indices ≈ [np.int64(1), np.int64(112), np.int64(223), np.int64(334), np.int64(445), np.int64(556), np.int64(667), np.int64(778), np.int64(889), np.int64(1000)])
  Population size   : 100
  Worker processes  : auto
  Output directory  : /Users/stijnkeukens/epa141a-main/results

[1/1] seed = 9845531
----------------------------------------
    workers = auto (cpu_count-1)


[MainProcess/INFO] pool started with 14 workers
50019it [6:19:31,  2.20it/s]                                                   
[MainProcess/INFO] optimization completed, found 10 solutions
[MainProcess/INFO] terminating pool


    convergence metrics  →  /Users/stijnkeukens/epa141a-main/results/UTILITARIAN_50000_9845531/convergence_9845531.csv
    10 Pareto solutions  →  /Users/stijnkeukens/epa141a-main/results/UTILITARIAN_50000_9845531/pareto_front_9845531.csv
    convergence archive  →  /Users/stijnkeukens/epa141a-main/results/UTILITARIAN_50000_9845531/UTILITARIAN_50000_9845531.tar.gz
    elapsed: 379.9 min

All 1 seeds done in 380 min (6.33 h).
Results → /Users/stijnkeukens/epa141a-main/results

Exit code: 0  (OK)


---

## Step 3 — Load and sanity-check the Pareto fronts

Each completed seed writes `results/UTILITARIAN_<nfe>_<seed>/` containing `pareto_front_<seed>.csv` (levers + the four objectives), `convergence_<seed>.csv` (epsilon-progress and operator probabilities, used in Assignment 6), and the ArchiveLogger `.tar.gz` (also Assignment 6). We load every Pareto front found and summarise it.


In [8]:
import glob

OBJECTIVE_COLS = ["welfare", "fraction_above_threshold",
                  "welfare_loss_damage", "welfare_loss_abatement"]

csv_files = sorted(glob.glob(
    os.path.join(RESULTS_ROOT, "**", "pareto_front_*.csv"), recursive=True))

if not csv_files:
    raise FileNotFoundError(
        f"No pareto_front_*.csv under {RESULTS_ROOT}. Run Step 2 first.")

dfs = []
for path in csv_files:
    seed_str = os.path.basename(path).replace("pareto_front_", "").replace(".csv", "")
    dir_name = os.path.basename(os.path.dirname(path))        # UTILITARIAN_<nfe>_<seed>
    parts    = dir_name.split("_")
    try:
        nfe = int(parts[-2])
    except (ValueError, IndexError):
        nfe = None
    df = pd.read_csv(path)
    df["seed"] = int(seed_str)
    df["nfe"]  = nfe
    dfs.append(df)
    print(f"  seed {seed_str:>10s}  NFE {str(nfe):>8s}: {len(df):4d} Pareto solutions")

all_results = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(all_results)} solutions across {len(dfs)} seed(s).")
print("\nObjective statistics (all four are lower-is-better):")
print(all_results[OBJECTIVE_COLS].describe().round(4))


  seed    9845531  NFE    50000:   10 Pareto solutions

Total: 10 solutions across 1 seed(s).

Objective statistics (all four are lower-is-better):
        welfare  fraction_above_threshold  welfare_loss_damage  \
count   10.0000                   10.0000              10.0000   
mean   103.5306                    0.7000            3635.3117   
std      0.0938                    0.3091              11.1755   
min    103.4585                    0.2000            3620.2780   
25%    103.4697                    0.4500            3628.8910   
50%    103.4897                    0.8000            3632.3140   
75%    103.5568                    0.9750            3642.0828   
max    103.7379                    1.0000            3653.8636   

       welfare_loss_abatement  
count                 10.0000  
mean               12840.9640  
std                 4767.1817  
min                10774.0545  
25%                10866.5422  
50%                11057.6253  
75%                11190.6837  
m

In [9]:
# A healthy front should have several distinct, non-degenerate solutions that
# actually beat business-as-usual (fraction_above_threshold < 1.0).
obj = all_results[OBJECTIVE_COLS]

checks = [
    ("More than 1 Pareto solution",                       len(obj) > 1),
    ("No NaN values in objectives",                       bool(obj.notna().all().all())),
    ("fraction_above_threshold spans > 0.05",
     (obj["fraction_above_threshold"].max() - obj["fraction_above_threshold"].min()) > 0.05),
    ("welfare_loss_abatement varies",                     obj["welfare_loss_abatement"].std() > 0),
    ("At least one policy beats BAU (frac_above < 1.0)",  bool((obj["fraction_above_threshold"] < 1.0).any())),
]

print("Non-triviality checks:")
for label, ok in checks:
    print(f"  {'OK ' if ok else 'XX '} {label}")

passed = sum(ok for _, ok in checks)
print(f"\n{passed}/{len(checks)} checks passed.")
print("Healthy front — proceed to build the reference set." if passed == len(checks)
      else "Some checks failed — increase NFE or verify the run.")


Non-triviality checks:
  OK  More than 1 Pareto solution
  OK  No NaN values in objectives
  OK  fraction_above_threshold spans > 0.05
  OK  welfare_loss_abatement varies
  OK  At least one policy beats BAU (frac_above < 1.0)

5/5 checks passed.
Healthy front — proceed to build the reference set.


---

## Step 4 — Build the combined reference set

This is the deliverable the rest of the pipeline depends on. We pool every seed's archive and reduce it to the **non-dominated** set across the four objectives (all four are lower-is-better, exactly as declared in Assignment 4), then save it as `results/reference_set_utilitarian.csv`. Assignments 6, 7 and 8 read this single file, so producing it here is what keeps the four notebooks consistent.

With one seed the pooled archive is already non-dominated, so the filter changes nothing; with the full five seeds it merges them. (The textbook refinement is epsilon-non-dominated merging on the same epsilon grid used during the search; a plain non-dominated filter is the correct special case and is used here to stay self-contained.)


In [10]:
def nondominated_min(frame, cols):
    """Keep rows not dominated by any other, treating every column as minimise."""
    X = frame[cols].to_numpy(dtype=float)
    n = len(X)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]:
            continue
        dominated_by_any = np.any(
            np.all(X <= X[i], axis=1) & np.any(X < X[i], axis=1)
        )
        keep[i] = not dominated_by_any
    return frame[keep].reset_index(drop=True)

# Decision-variable columns = everything that is NOT an objective or a bookkeeping
# column. run_optimization_local.py does not prefix the 244 RBF levers with
# "lever_", so we detect them by exclusion rather than by name.
META_COLS  = ["seed", "nfe"]
lever_cols = [c for c in all_results.columns
              if c not in set(OBJECTIVE_COLS) | set(META_COLS)
              and not str(c).startswith("Unnamed")]

assert lever_cols, (
    "No decision-variable columns detected. Columns present: "
    f"{list(all_results.columns)}"
)
print(f"Detected {len(lever_cols)} decision-variable columns "
      f"(e.g. {lever_cols[:3]}{' ...' if len(lever_cols) > 3 else ''})")

keep_cols = lever_cols + OBJECTIVE_COLS

# Pool -> drop exact duplicate policies -> non-dominated filter
pooled = all_results[keep_cols].drop_duplicates(subset=lever_cols).reset_index(drop=True)
reference_set = nondominated_min(pooled, OBJECTIVE_COLS)

ref_path = os.path.join(RESULTS_ROOT, "reference_set_utilitarian.csv")
reference_set.to_csv(ref_path, index=False)

print(f"Pooled (deduplicated) : {len(pooled)} policies")
print(f"Reference set         : {len(reference_set)} non-dominated policies")
print(f"Decision variables    : {len(lever_cols)} lever columns")
print(f"Saved -> {ref_path}")
print("\nReference-set objective ranges:")
print(reference_set[OBJECTIVE_COLS].describe().round(4))


Detected 244 decision-variable columns (e.g. ['center_0', 'center_1', 'center_2'] ...)
Pooled (deduplicated) : 10 policies
Reference set         : 10 non-dominated policies
Decision variables    : 244 lever columns
Saved -> /Users/stijnkeukens/epa141a-main/results/reference_set_utilitarian.csv

Reference-set objective ranges:
        welfare  fraction_above_threshold  welfare_loss_damage  \
count   10.0000                   10.0000              10.0000   
mean   103.5306                    0.7000            3635.3117   
std      0.0938                    0.3091              11.1755   
min    103.4585                    0.2000            3620.2780   
25%    103.4697                    0.4500            3628.8910   
50%    103.4897                    0.8000            3632.3140   
75%    103.5568                    0.9750            3642.0828   
max    103.7379                    1.0000            3653.8636   

       welfare_loss_abatement  
count                 10.0000  
mean         

### Reading the front through the Japan & South Korea mandate

The reference set is a menu of trade-offs, not a recommendation, and the shape of that menu is what matters for our negotiating position:

- **`fraction_above_threshold` vs `welfare_loss_abatement`** is the central tension. Policies that drive the 2°C exceedance fraction toward its minimum sit at the high-abatement end of the front. Our mandate is deliberately **overshoot-tolerant**, so unlike a vulnerable-island actor we are *not* obliged to chase the extreme climate end; a policy that allows a bounded, temporary overshoot while keeping abatement efficient is fully consistent with our case.
- **`welfare_loss_damage`** is the quantity that tells us whether an overshoot stays *bounded for the exposed regions* (`rcam`, `rjan57`). It is the figure our prioritarian, worst-off lens will scrutinise and the basis for any adaptation-finance offer.
- **`welfare`** (the Utilitarian efficiency loss) is the lens we optimised on and the spine of our efficiency argument.

Which point on this front we actually defend depends on how these trade-offs hold up **across** scenarios, not just under SSP2-4.5. That is exactly what Assignment 8 tests with satisficing and minimax-regret, so we carry the whole reference set forward rather than committing to a single policy now.


---

## Reflection Questions

**1. Multiple seeds.** Why run the MOEA with 5 different random seeds rather than one long seed at 5× the NFE? What property of the algorithm makes this necessary?

MOEAs are stochastic: the starting population and the random application of variation operators mean different seeds explore different parts of a 244-dimensional, non-convex landscape and can settle on different stretches of the Pareto front. A single long run can entrench itself near one region and never sample others, and simply giving it 5× the NFE does not fix that — it deepens one search rather than diversifying it. Five independent seeds, each starting fresh (including GenerationalBorg's operator probabilities, which re-learn per seed), and then pooled by epsilon-dominance, give a more complete and more trustworthy reference set. The single-seed smoke test in this notebook is only a pipeline check; the reference set we submit for Japan & South Korea should be built from the full five-seed run.

**2. NFE size.** What diagnostic would tell you whether the NFE you used was enough?

The convergence diagnostics developed in Assignment 6 — epsilon-progress and hypervolume against the combined reference set. If both have flattened well before the final evaluation, the budget was sufficient and extra NFE buys little; if either is still climbing at the end, the front is incomplete and the budget was too small. For our purposes an unconverged front is not just a technical worry: it could miss the efficient, overshoot-tolerant policies in the interior of the front that best fit our mandate, biasing the menu toward whatever region the search happened to reach first.

**3. Operator adaptation.** GenerationalBorg adapts the selection probabilities of its six operators (SBX, PCX, DE, UNDX, SPX, UM) during the run. What does that mean and why does it help in a 244-dimensional RBF space?

Every operator starts equally likely. After each generation the algorithm records which operators produced archive-improving offspring and shifts probability toward them. In 244 dimensions no single operator dominates: SBX refines near existing solutions, DE exploits correlations between variables, UM injects diversity when the search stalls, and the best mix changes as the run moves from broad exploration to local refinement. Letting the algorithm re-weight operators on the fly means it self-tunes to the geometry of the JUSTICE problem instead of relying on a fixed, manually chosen operator that would be wrong for part of the run.


---

## Appendix — Script reference

```
run_optimization_local.py — argument list
  --nfe           int     NFE per seed                 (default: 50000)
  --seeds         int+    Random seeds                 (default: all 5)
  --output_dir    str     Results root directory        (here: ../results)
  --n_processes   int     Worker processes              (default: auto)
  --n_ensembles   int     FaIR ensemble members         (default: 10)
  --config        str     JSON config path              (here: ../config/config_student.json)
  --population    int     MOEA population size          (default: 100)
```

**Output layout (consumed by Assignments 6–8):**

```
results/
├── UTILITARIAN_<nfe>_<seed>/
│   ├── UTILITARIAN_<nfe>_<seed>.tar.gz   <- convergence archive (Assignment 6)
│   ├── convergence_<seed>.csv            <- epsilon-progress + operator probs (Assignment 6)
│   └── pareto_front_<seed>.csv           <- this seed's Pareto front
└── reference_set_utilitarian.csv         <- pooled non-dominated set (Assignments 6, 7, 8)
```
